<a href="https://colab.research.google.com/github/lnittala1-ops/BAN482/blob/main/BAN482_Data_Preprocessing_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Data pre-processing using LLMs

In [ ]:
# Install required libraries
!pip install -q instructor openai

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Here is a sample dataframe
product_review_data = {
    'review_id': ['REV-1042', 'REV-1043', 'REV-1044'],
    'date_submitted': ['2023-10-25', '2023-10-26', '2023-10-26'],
    'user_id': ['U-8472', 'U-1928', 'U-5531'],
    'source': ['Mobile App', 'Web Browser', 'Mobile App'],
    'review_text': ["I bought the Dyson V15 yesterday. It's amazing, 10/10!",
    "My iPhone 12 screen cracked immediately. I hate it. I want my money back.",
    "The Nike Air Max shoes are okay, a bit tight but they look nice."]
}

# Create the Pandas DataFrame
df = pd.DataFrame(product_review_data)
df

In [ ]:
import pandas as pd
from pydantic import BaseModel, Field
import instructor
from openai import OpenAI

#### STEP 1: Connecting to the AI model

In [ ]:
"""
Ollama follows the OpenAI standard. So we will use the openai library.
'instructor' sits on top of this connection to enable data extraction.
"""
client = instructor.from_openai(
    OpenAI(
        # We redirect the client to look at your own computer instead of the internet
        base_url="http://localhost:11434/v1",
        # The library requires a key, but because we are local, Ollama ignores it
        api_key="ollama",
    ),
    # JSON mode forces the AI to output computer-readable text, not chatty paragraphs
    mode=instructor.Mode.JSON
)


#### STEP 2: Defining the Schema & Prompt Engineering

In [ ]:
"""
We don't need to write a long, complex text prompt.
Instead, we use Pydantic's `Field` to write our instructions directly into the columns.
'Instructor' will read these descriptions and pass them to the AI so it knows
EXACTLY how to analyze and format the data.
"""
class ReviewData(BaseModel):
    product_name: str = Field(
        description="The name of the physical product being reviewed."
    )
    sentiment_score: int = Field(
        description="A score from 1 to 10 based on the user's tone. 10 is extremely positive, 1 is extremely negative."
    )
    review_tone: str = Field(
        description="Tone of the review. Positive, Negative, or Neutral"
    )

#### STEP 3: Setting up the Extraction Loop

In [ ]:
"""
We process the data one piece of text at a time. We send the text AND our blueprint
to the AI, and it sends back the structured data.
"""

# We create an empty list to hold the clean data as it comes back from the AI
extracted_rows = []
print("Extracting data with Gemma4:e2b ...\n")

for text in df["review_text"]:
    extracted_info = client.chat.completions.create(
        # Specify the exact local model you downloaded via Ollama
        model="gemma4:e2b",
        messages=[
            # The 'system' role gives the AI its overarching job description
            {"role": "system", "content": "You are a precise data extraction assistant."},
            # The 'user' role is the specific data we want it to process right now
            {"role": "user", "content": text}
        ],
        # MAGIC ALERT: This tells the AI it MUST use the blueprint we defined in Step 2!
        response_model=ReviewData,

        # --- REPRODUCIBILITY LOCKDOWN ---
        # We want the AI output to be deterministic and consistent
        # Temperature 0 removes creativity, and Seed 42 gives it a fixed mathematical starting point.
        temperature=0.0,
        seed=42
    )
    # 'extracted_info' is currently a special Pydantic object.
    # .model_dump() translates it back into a standard Python dictionary so Pandas can read it.
    extracted_rows.append(extracted_info.model_dump())

#### STEP 4: Integration with Data Analytics (Pandas)

In [ ]:
"""
Because we forced the AI to give us a clean list of dictionaries,
Pandas can instantly convert it into a DataFrame without any extra cleaning.
"""
df = pd.DataFrame(extracted_rows)

print("--- The Cleaned Tabular Data ---")
print(df)